# EN.737 — GPT-2 inference: KV caching and batching (Q3b)
Measure the effect of KV caching and batch size on German → English generation using GPT-2 small only. This notebook does not train any models or require M2's model.

**Quick start:** select a GPU runtime; run setup; set `PROFILE="smoke"` for a short pretrained-model check (no shared data required). For report results use `PROFILE="project"`, point `CHECKPOINT` to Notebook 1's `best_model_for_inference` directory and `DATA_DIR` to the shared split folder (or `DATA_ZIP`). Use Drive to retain outputs. Do not present smoke results as fine-tuned results.

Primary experiment: greedy decoding of the same 32 test sources, with a common padded prompt width and exactly 32 or 64 generated tokens, at batch sizes 1, 2, 4, 8, 16, 32, cache on/off, three repeats. The fixed-length benchmark intentionally disables EOS/newline stopping to isolate computation; it is **not** a translation-quality benchmark. Optional natural decoding uses the same stop convention as Notebook 1. Timing excludes tokenization/transfers, warmup, and output inspection; it includes prefill and generation. The exact sequence of configurations is shuffled per repeat.

In [ ]:
import sys, subprocess, importlib.metadata as im
PINS = {'transformers':'4.51.3', 'sacrebleu':'2.5.1'}
missing=[]
for p,v in PINS.items():
    try: installed=im.version(p)
    except im.PackageNotFoundError: installed=None
    if installed!=v: missing.append(f'{p}=={v}')
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('Dependencies ready. If Transformers was imported before installation, restart the runtime.')

## 1. Configuration and hypotheses (record before benchmarking)
Precision is FP32, matching M0's measurements and the report's 72 KiB figure. The course expects a quantitative prediction, repeated comparisons, and a link to cache memory. Proposed predictions (not results): at batch size 1 and 64 generated tokens, caching gives at least 1.5× speedup; cached throughput at batch size 8 is at least 2× batch size 1; the GPU memory retained after a cached call, minus the returned token IDs, is within 10% of the predicted cache size $2\,n_{layer}\,d_{model}$ values per cached position (72 KiB per position in FP32), for every cached configuration. The K/V tensor payload is also checked against the formula, but only as an implementation check: it counts tensor shapes, so it cannot fail as a prediction. These thresholds are hypotheses, not guarantees. Peak allocated memory may go either way because uncached decoding recomputes larger intermediates. The primary latency metric is synchronous batch-call wall time; throughput is total generated tokens divided by total wall time over a fixed workload. Report peak allocated memory, not CUDA allocator reserved memory, as the primary memory metric.

In [ ]:
import csv, gc, hashlib, json, math, os, platform, random, time, zipfile
import importlib.metadata as im
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
from huggingface_hub import HfApi
from IPython.display import display

PROFILE = 'project'          # project or smoke
USE_DRIVE = 'auto'           # 'auto': mount Drive in Colab, fall back to runtime disk if mounting fails (e.g. Colab VS Code extension); True: require Drive; False: never mount
CHECKPOINT = '/content/drive/MyDrive/en737_m1_runs/project-fc6cf65a02ca/best_model_for_inference'  # Notebook 1 export (full FT, seed 42, epoch 3)
DATA_DIR = '/content/team_data' if Path('/content').exists() else './team_data'
DATA_ZIP = ''                # optional M2 archive containing shared_splits/test.tsv
OUTPUT_ROOT = '/content/en737_kv_runs' if Path('/content').exists() else './en737_kv_runs'
BASE_MODEL = 'openai-community/gpt2'
BASE_REVISION = None         # smoke only; resolve exact SHA when unspecified
PRECISION = 'fp32'           # fp32 matches M0 and the report (72 KiB per cached position); fp16/bf16 halve it
ATTENTION = 'eager'          # fixed attention implementation across both conditions
BATCH_SIZES = [1,2,4,8,16,32]
GEN_LENGTHS = [32,64]
N_SENTENCES = 32             # must be divisible by every batch size
REPEATS = 3
WARMUPS = 1                  # per configuration, before measured repetitions
SEED = 17
RUN_NATURAL = False          # optional extra sweep with real stopping, cap 128
NATURAL_CAP = 128
if USE_DRIVE == 'auto': USE_DRIVE = None if Path('/content').exists() else False
assert USE_DRIVE in {None, True, False}, "USE_DRIVE must be 'auto', True or False"
if USE_DRIVE is not False:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        OUTPUT_ROOT = '/content/drive/MyDrive/en737_kv_runs'
        USE_DRIVE = True
        # Set CHECKPOINT and DATA_DIR above to their actual Drive paths.
    except Exception as e:
        if USE_DRIVE: raise
        USE_DRIVE = False
        print(f'Drive not mounted ({type(e).__name__}: {e}). Saving to {OUTPUT_ROOT} on the runtime disk; '
              'download the report zip before the runtime disconnects.')
if PROFILE=='smoke':
    BATCH_SIZES=[1,2]; GEN_LENGTHS=[4]; N_SENTENCES=4; REPEATS=2; RUN_NATURAL=False
assert PROFILE in {'project','smoke'} and PRECISION in {'fp16','bf16','fp32'}
assert all(b>0 and N_SENTENCES%b==0 for b in BATCH_SIZES) and min(GEN_LENGTHS)>0
assert REPEATS>=2 and WARMUPS>=1
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type!='cuda' and PROFILE=='project': raise RuntimeError('Select a GPU runtime for report benchmarks.')
DTYPE={'fp16':torch.float16,'bf16':torch.bfloat16,'fp32':torch.float32}[PRECISION]
if DEVICE.type=='cpu': DTYPE=torch.float32; torch.set_num_threads(min(4,torch.get_num_threads()))
if DTYPE==torch.bfloat16 and DEVICE.type=='cuda': assert torch.cuda.is_bf16_supported()
if DEVICE.type=='cuda': torch.backends.cuda.matmul.allow_tf32=False
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if DEVICE.type=='cuda': torch.cuda.manual_seed_all(SEED)
PREDICTION=dict(speedup_B1_G64_min=1.5, cached_throughput_B8_over_B1_G64_min=2.0,
                retained_memory_relative_error_max=0.10)
MIB=2**20

def dump(path,obj):
    Path(path).write_text(json.dumps(obj,indent=2,ensure_ascii=False,default=str,allow_nan=False),encoding='utf-8')
def sha_bytes(b): return hashlib.sha256(b).hexdigest()
def sha_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(2**20),b''): h.update(chunk)
    return h.hexdigest()
def sync():
    if DEVICE.type=='cuda': torch.cuda.synchronize(DEVICE)
def cleanup():
    gc.collect()
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
print('Device:',DEVICE, '| dtype:',DTYPE, '| profile:',PROFILE)

## 2. Load the checkpoint and fixed workload
Notebook 1 exports standard GPT-2 weights (LoRA merged when selected) and tokenizer. Project mode requires that exact local checkpoint rather than silently falling back to the pretrained model. The test sample is selected by a fixed seed; all configurations see the same sources in the same order. All prompts are left padded to a single common width so batch size is not confounded by padding length. Padded positions still occupy cache memory, and attention masks prevent them from contributing to the predictions. This isolates batch scaling, though it is not a length-bucketed production workload.

In [ ]:
if CHECKPOINT:
    path=Path(CHECKPOINT).expanduser()
    if not (path/'config.json').exists(): raise FileNotFoundError(f'Missing GPT-2 checkpoint: {path}')
    weights=sorted(p for p in path.iterdir() if p.suffix in {'.safetensors','.bin'} and p.is_file())
    if not weights: raise FileNotFoundError('Expected standalone model weights exported by Notebook 1.')
    selected_path=path.parent/'selected_model.json'
    selected=json.loads(selected_path.read_text()) if selected_path.exists() else None
    model_source=str(path.resolve()); revision=None
    checkpoint_info=dict(path=model_source, selected_model=selected,
                         files={p.name:sha_file(p) for p in sorted(path.iterdir()) if p.is_file()})
else:
    if PROFILE=='project': raise ValueError('Set CHECKPOINT to Notebook 1 best_model_for_inference. Use smoke for pretrained testing.')
    model_source=BASE_MODEL; revision=BASE_REVISION or HfApi().model_info(BASE_MODEL).sha
    checkpoint_info=dict(model=BASE_MODEL,revision=revision,pretrained_only=True)
model=AutoModelForCausalLM.from_pretrained(model_source,revision=revision,torch_dtype=DTYPE,
                                         attn_implementation=ATTENTION).to(DEVICE).eval()
tokenizer=AutoTokenizer.from_pretrained(model_source,revision=revision,use_fast=True)
assert model.config.model_type=='gpt2'
LAYERS=model.config.n_layer; WIDTH=model.config.n_embd; HEADS=model.config.n_head
assert (LAYERS,WIDTH,HEADS)==(12,768,12), 'This benchmark is scoped to GPT-2 small.'
EOS=tokenizer.eos_token_id; PAD=EOS; tokenizer.pad_token=tokenizer.eos_token; tokenizer.padding_side='left'
model.config.pad_token_id=PAD
# Same stop tokens as Notebook 1, only for optional natural generation.
STOP_IDS=sorted({EOS}|{i for i in range(len(tokenizer)) if '\n' in tokenizer.decode([i],clean_up_tokenization_spaces=False)})
STOP_SET=set(STOP_IDS)
if PROFILE=='smoke':
    sources=['Guten Morgen.', 'Ich bin müde.', 'Wie geht es Ihnen?', 'Das ist ein Buch.']
    rows=[dict(id=str(i),source=s,target='') for i,s in enumerate(sources)]
    data_hash=sha_bytes(json.dumps(rows,ensure_ascii=False).encode())
else:
    if DATA_ZIP:
        with zipfile.ZipFile(DATA_ZIP) as z:
            candidates=[x for x in z.namelist() if x.endswith('shared_splits/test.tsv') or x=='test.tsv']
            if len(candidates)!=1: raise ValueError(f'Expected one test.tsv: {candidates}')
            raw=z.read(candidates[0])
    else:
        root=Path(DATA_DIR).expanduser(); path=root/'test.tsv'
        if not path.exists():
            candidates=list(root.rglob('shared_splits/test.tsv'))
            if len(candidates)!=1: raise FileNotFoundError('Set DATA_DIR to the shared_splits folder, or DATA_ZIP.')
            path=candidates[0]
        raw=path.read_bytes()
    data_hash=sha_bytes(raw)
    reader=csv.DictReader(raw.decode('utf-8-sig').splitlines(keepends=True),delimiter='\t')
    assert {'source','target'}.issubset(reader.fieldnames or [])
    rows=[dict(id=str(r.get('id') or i),source=r['source'],target=r['target']) for i,r in enumerate(reader)]
assert len({r['id'] for r in rows})==len(rows) and len(rows)>=N_SENTENCES
indices=random.Random(SEED).sample(range(len(rows)),N_SENTENCES)
workload=[rows[i] for i in indices]
prompts=[f"German: {r['source']}\nEnglish:" for r in workload]
encoded=tokenizer(prompts,add_special_tokens=False,padding=True,return_tensors='pt')
PROMPT_WIDTH=encoded['input_ids'].shape[1]
assert PROMPT_WIDTH+max(GEN_LENGTHS+([NATURAL_CAP] if RUN_NATURAL else []))<=model.config.n_positions
# Keep fixed input on CPU, transfer batch to GPU outside each timed interval.
INPUT_IDS=encoded['input_ids']; MASK=encoded['attention_mask']
for i,r in enumerate(workload): r['prompt_tokens']=int(MASK[i].sum())
print('Prompt width:',PROMPT_WIDTH,'| actual lengths:',min(r['prompt_tokens'] for r in workload),max(r['prompt_tokens'] for r in workload))
print('Parameters:',sum(p.numel() for p in model.parameters()))

In [ ]:
# Persist configuration before any timed generation (including warmups).
software={p:im.version(p) for p in ['torch','transformers','sacrebleu','numpy','pandas','tokenizers']}
hardware=dict(device=str(DEVICE),gpu=torch.cuda.get_device_name(DEVICE) if DEVICE.type=='cuda' else None,
              gpu_total_bytes=torch.cuda.get_device_properties(DEVICE).total_memory if DEVICE.type=='cuda' else None,
              cuda=torch.version.cuda,python=platform.python_version())
experiment=dict(profile=PROFILE,checkpoint=checkpoint_info,data_sha256=data_hash,ids=[r['id'] for r in workload],
                batch_sizes=BATCH_SIZES,generated_lengths=GEN_LENGTHS,n_sentences=N_SENTENCES,repeats=REPEATS,
                warmups=WARMUPS,seed=SEED,dtype=str(DTYPE),attention=ATTENTION,prompt_width=PROMPT_WIDTH,
                natural=RUN_NATURAL,natural_cap=NATURAL_CAP,predictions=PREDICTION,software=software,hardware=hardware)
RUN_ID=sha_bytes(json.dumps(experiment,sort_keys=True).encode())[:12]
# Distinct output folder per invocation, so reruns never overwrite earlier measurements.
OUT=Path(OUTPUT_ROOT)/f'{PROFILE}-{RUN_ID}-{datetime.now(timezone.utc):%Y%m%dT%H%M%S%f}'
OUT.mkdir(parents=True,exist_ok=False)
dump(OUT/'preregistration.json',dict(recorded_at=datetime.now(timezone.utc).isoformat(),**experiment))
pd.DataFrame(workload).to_csv(OUT/'workload.csv',index=False)
print('Results directory:',OUT)

## 3. Cache theory and measurement
For GPT-2's standard multi-head attention, keys and values each have shape `(B, heads, T, head_dim)` per layer. The predicted payload is

$$M_{KV}=2 B L T d_{model} s,$$

where $s$ is bytes per cache scalar. GPT-2 small uses 36 KiB per batch element per cached position in FP16/BF16, or 72 KiB in FP32. A generation call returning $G$ tokens normally caches $P+G-1$ positions: the last generated token has not been fed through the model yet. Verify the observed length rather than assume. The prediction is tested against retained CUDA allocated memory after the call, minus the returned sequence IDs; the actual K/V tensor payload (`numel × element_size`) is checked against the formula only to confirm the cache has the expected shape and dtype. Retained memory also includes returned sequence IDs and possibly other allocations; peak memory includes transient activations/workspace, model weights and inputs. No-cache peak memory is not necessarily lower.

In [ ]:
def predicted_bytes(b,t,scalar_bytes): return 2*b*LAYERS*t*WIDTH*scalar_bytes

def cache_info(cache,b):
    if cache is None: return dict(cache_bytes=0,cache_length=0,cache_scalar_bytes=None,cache_dtype=None)
    if hasattr(cache,'layers') and not isinstance(cache,(tuple,list)):
        pairs=[(layer.keys,layer.values) for layer in cache.layers]
    elif hasattr(cache,'key_cache'):
        pairs=list(zip(cache.key_cache,cache.value_cache))
    elif hasattr(cache,'to_legacy_cache'):
        pairs=cache.to_legacy_cache()
    else:
        pairs=cache
    assert len(pairs)==LAYERS, f'Unexpected cache layers: {len(pairs)}'
    tensors=[]
    for pair in pairs:
        assert len(pair)==2, 'Expected GPT-2 self-attention K/V only.'
        tensors.extend(pair)
    t=tensors[0].shape[-2]; s=tensors[0].element_size(); dtype=tensors[0].dtype
    assert all(x.shape==(b,HEADS,t,WIDTH//HEADS) and x.dtype==dtype for x in tensors)
    return dict(cache_bytes=sum(x.numel()*x.element_size() for x in tensors),cache_length=t,
                cache_scalar_bytes=s,cache_dtype=str(dtype))

# Unit checks: formula and legacy cache extraction on tiny synthetic tensors (no GPU allocations).
assert predicted_bytes(1,1,2)==36*1024
assert predicted_bytes(1,1,4)==72*1024
fake=tuple((torch.empty(2,HEADS,3,WIDTH//HEADS,dtype=torch.float16),)*2 for _ in range(LAYERS))
assert cache_info(fake,2)['cache_bytes']==predicted_bytes(2,3,2)
del fake
print('Theoretical payload checks passed.')

## 4. Controlled generation and synchronized timing
Use `model.generate` with a fresh `GenerationConfig` for every call to avoid inherited sampling/beam settings. Cache enabled/disabled is the only decoding difference for a given batch and length. A fresh generation call starts with no past cache; warmups are excluded. GPU synchronization before and after the call makes wall time comparable. Inputs already reside on the GPU when timing starts; copying output IDs to CPU and inspecting the cache happen after timing. Measure `memory_allocated` and `max_memory_allocated` (bytes actually allocated by PyTorch), not `nvidia-smi` or reserved memory. Cache memory is measured before deleting the returned cache. Compare generated IDs across settings; floating point differences can alter greedy decisions, and any mismatches are recorded rather than hidden.

In [ ]:
@torch.inference_mode()
def timed_call(start,b,g,use_cache,mode):
    ids=INPUT_IDS[start:start+b].to(DEVICE); mask=MASK[start:start+b].to(DEVICE)
    cfg=GenerationConfig(do_sample=False,num_beams=1,max_new_tokens=g,
                         min_new_tokens=g if mode=='fixed' else 0,
                         eos_token_id=None if mode=='fixed' else STOP_IDS,
                         pad_token_id=PAD,bos_token_id=EOS,use_cache=use_cache,
                         return_dict_in_generate=True,output_scores=False,
                         output_attentions=False,output_hidden_states=False)
    sync()
    if DEVICE.type=='cuda':
        baseline=torch.cuda.memory_allocated(DEVICE)
        baseline_req=torch.cuda.memory_stats(DEVICE).get('requested_bytes.all.current')
        torch.cuda.reset_peak_memory_stats(DEVICE)
    else: baseline=None
    tic=time.perf_counter()
    output=model.generate(input_ids=ids,attention_mask=mask,generation_config=cfg)
    sync(); elapsed=time.perf_counter()-tic
    if DEVICE.type=='cuda':
        peak=torch.cuda.max_memory_allocated(DEVICE); retained=torch.cuda.memory_allocated(DEVICE)-baseline
        reserved=torch.cuda.max_memory_reserved(DEVICE)
        now_req=torch.cuda.memory_stats(DEVICE).get('requested_bytes.all.current')
        retained_req=now_req-baseline_req if None not in (now_req,baseline_req) else None
    else: peak=retained=reserved=retained_req=None
    # No GPU allocation-heavy work before measuring peak / retained.
    ci=cache_info(getattr(output,'past_key_values',None),b)
    if use_cache and ci['cache_bytes']==0: raise RuntimeError('Cache enabled but no K/V returned; inspect Transformers version.')
    if not use_cache: assert ci['cache_bytes']==0
    seq=output.sequences; steps=seq.shape[1]-PROMPT_WIDTH
    if mode=='fixed': assert steps==g
    if use_cache: assert ci['cache_length']==PROMPT_WIDTH+steps-1, (ci,steps)
    # For natural decoding count tokens through first stop, not pad/idle slots.
    generated=seq[:,PROMPT_WIDTH:].cpu().tolist()
    trimmed=[s if mode=='fixed' else s[:next((j+1 for j,t in enumerate(s) if t in STOP_SET),len(s))] for s in generated]
    useful=sum(map(len,trimmed)); slots=b*steps
    pred=predicted_bytes(b,ci['cache_length'],ci['cache_scalar_bytes']) if use_cache else 0
    seq_bytes=seq.numel()*seq.element_size()
    result=dict(start=start,batch_size=b,mode=mode,max_new_tokens=g,use_cache=use_cache,
                latency_s=elapsed,generated_tokens=useful,decode_slots=slots,decode_steps=steps,
                baseline_bytes=baseline,peak_bytes=peak,peak_increment_bytes=peak-baseline if peak is not None else None,
                reserved_peak_bytes=reserved,retained_increment_bytes=retained,
                returned_sequence_bytes=seq_bytes,predicted_cache_bytes=pred,**ci)
    result['retained_minus_ids_bytes']=retained-seq_bytes if retained is not None else None
    # Requested bytes exclude the caching allocator's block rounding/reuse; diagnostic only.
    result['requested_minus_ids_bytes']=retained_req-seq_bytes if retained_req is not None else None
    del output,seq,ids,mask
    return result,trimmed

# Smoke sanity: same input / fixed length, cache on/off. Not included in final benchmark.
# Project configuration already saved before this check.
check_g=min(GEN_LENGTHS)
check_on,tok_on=timed_call(0,1,check_g,True,'fixed')
check_off,tok_off=timed_call(0,1,check_g,False,'fixed')
print('Sanity check: generated tokens',check_on['generated_tokens'], '| exact ID agreement:',tok_on==tok_off,
      '| K/V payload:',check_on['cache_bytes'],'bytes')

## 5. Run the sweep
Each configuration runs the full fixed workload in batches; the same 32 sentences are used even when batch size changes. Repeat order is randomized deterministically. Warm each configuration before measuring. CSV files are updated after each configuration; out-of-memory conditions are recorded and skipped rather than silently dropping a batch size or changing the workload. Do not compare incomplete configurations. A repeat is the experimental unit for timing variability; individual batches within a repeat are not independent repeats. The natural-decoding sweep, if enabled, is kept separate from fixed-length results.

In [ ]:
configs=[('fixed',g,b,c) for g in GEN_LENGTHS for b in BATCH_SIZES for c in [False,True]]
if RUN_NATURAL: configs += [('natural',NATURAL_CAP,b,c) for b in BATCH_SIZES for c in [False,True]]
raw_rows=[]; pred_rows=[]; status_rows=[]; failed=set()
def keystr(k): return f'{k[0]}-G{k[1]}-B{k[2]}-cache{int(k[3])}'
def persist():
    pd.DataFrame(raw_rows).to_csv(OUT/'batch_measurements.csv',index=False)
    pd.DataFrame(pred_rows).to_csv(OUT/'generated_ids.csv',index=False)
    pd.DataFrame(status_rows).to_csv(OUT/'run_status.csv',index=False)

def execute_config(k,repeat):
    mode,g,b,c=k; records=[]; predictions=[]
    for start in range(0,N_SENTENCES,b):
        rec,seqs=timed_call(start,b,g,c,mode); rec['repeat']=repeat; records.append(rec)
        for j,seq in enumerate(seqs):
            predictions.append(dict(mode=mode,max_new_tokens=g,batch_size=b,use_cache=c,repeat=repeat,
                                    id=workload[start+j]['id'],token_ids=json.dumps(seq),
                                    generated_tokens=len(seq)))
    return records,predictions

# All shapes warm once; no cache clearing between timed calls. Cleanup is outside timing.
for k in configs:
    mode,g,b,c=k
    try:
        for _ in range(WARMUPS): timed_call(0,b,g,c,mode)
        status_rows.append(dict(config=keystr(k),repeat=-1,status='warmup_ok'))
    except torch.cuda.OutOfMemoryError:
        failed.add(k); status_rows.append(dict(config=keystr(k),repeat=-1,status='OOM_warmup'))
    if k in failed: cleanup()
    persist()
for repeat in range(REPEATS):
    order=configs.copy(); random.Random(SEED+repeat).shuffle(order)
    for k in order:
        if k in failed: continue
        try:
            records,predictions=execute_config(k,repeat)
            raw_rows.extend(records); pred_rows.extend(predictions)
            status_rows.append(dict(config=keystr(k),repeat=repeat,status='ok'))
            print(f'{repeat+1}/{REPEATS} {keystr(k)}: {sum(r["latency_s"] for r in records):.2f}s')
        except torch.cuda.OutOfMemoryError:
            failed.add(k); status_rows.append(dict(config=keystr(k),repeat=repeat,status='OOM_measurement'))
            print('OOM:',keystr(k))
        if k in failed: cleanup()
        persist()
print('Completed:',len(raw_rows),'batch calls. Failed configurations:',[keystr(k) for k in failed])

## 6. Aggregate results and variability
For each complete repeat, batch latency is the mean call time; throughput is the total number of useful generated tokens divided by total call time. Also report amortized milliseconds per sentence, sentences/s, decode-slot throughput (includes finished/padded rows in natural mode), and peak allocated memory (maximum across batches). The table summarizes mean and sample SD across repeats. Missing/OOM configurations remain explicit in `run_status.csv`; configurations without all repeats are excluded from the primary table.

In [ ]:
raw=pd.DataFrame(raw_rows); pred_df=pd.DataFrame(pred_rows)
if raw.empty: raise RuntimeError('No completed measurements. Review run_status.csv.')
keys=['mode','max_new_tokens','batch_size','use_cache']

def mib_or_nan(v): return float(v)/MIB if pd.notna(v) else np.nan

def aggregate_repeats(frame):
    rr=[]
    for k,g in frame.groupby(keys+['repeat'],sort=True):
        b=int(k[2]); assert len(g)*b==N_SENTENCES and g.start.nunique()==len(g)
        total=g.latency_s.sum()
        rr.append(dict(zip(keys+['repeat'],k))|dict(
            batch_latency_ms=1000*g.latency_s.mean(),amortized_ms_per_sentence=1000*total/N_SENTENCES,
            tokens_per_s=g.generated_tokens.sum()/total,slots_per_s=g.decode_slots.sum()/total,
            sentences_per_s=N_SENTENCES/total,peak_mib=mib_or_nan(g.peak_bytes.max()),
            peak_increment_mib=mib_or_nan(g.peak_increment_bytes.max()),
            retained_mib=mib_or_nan(g.retained_increment_bytes.mean()),
            cache_mib=g.cache_bytes.mean()/MIB,generated_tokens=int(g.generated_tokens.sum())))
    return pd.DataFrame(rr)
repeat_df=aggregate_repeats(raw); repeat_df.to_csv(OUT/'repeat_metrics.csv',index=False)
metrics=['batch_latency_ms','amortized_ms_per_sentence','tokens_per_s','slots_per_s','sentences_per_s',
         'peak_mib','peak_increment_mib','retained_mib','cache_mib','generated_tokens']
summary=repeat_df.groupby(keys)[metrics].agg(['mean','std']).reset_index()
summary.columns=['_'.join(str(x) for x in col if x!='') for col in summary.columns]
counts=repeat_df.groupby(keys).size().rename('n_repeats').reset_index()
summary=summary.merge(counts,on=keys)
summary['complete']=summary.n_repeats.eq(REPEATS)
summary.to_csv(OUT/'summary_all.csv',index=False)
complete=summary[summary.complete].copy(); complete.to_csv(OUT/'summary_complete.csv',index=False)
if complete.empty: raise RuntimeError('No configuration completed all repeats.')
show=keys+['n_repeats','batch_latency_ms_mean','batch_latency_ms_std','tokens_per_s_mean','tokens_per_s_std',
           'peak_mib_mean','peak_mib_std','cache_mib_mean']
display(complete[show].round(2))

In [ ]:
# Check greedy output agreement: pair identical sentence IDs and repeat, not row order.
join=['mode','max_new_tokens','batch_size','repeat','id']
a=pred_df[pred_df.use_cache].merge(pred_df[~pred_df.use_cache],on=join,suffixes=('_on','_off'),validate='one_to_one')
a['exact_match']=a.token_ids_on.eq(a.token_ids_off)
if len(a):
    agreement=a.groupby(keys[:-1]+['repeat']).exact_match.agg(['mean','count']).reset_index()
    agreement.to_csv(OUT/'cache_output_agreement.csv',index=False)
    display(agreement)
    a[~a.exact_match].to_csv(OUT/'cache_output_mismatches.csv',index=False)
else: print('No matched cache on/off output pairs.')
# Across batch sizes, compare against batch 1 within each cache mode.
base=pred_df[pred_df.batch_size==1][['mode','max_new_tokens','use_cache','repeat','id','token_ids']]
bcmp=pred_df.merge(base,on=['mode','max_new_tokens','use_cache','repeat','id'],suffixes=('','_b1'),validate='many_to_one')
bcmp['exact_match']=bcmp.token_ids.eq(bcmp.token_ids_b1)
bcmp.groupby(keys+['repeat']).exact_match.agg(['mean','count']).reset_index().to_csv(OUT/'batch_output_agreement.csv',index=False)
# Optional natural generation: quality check (one repeat, per complete configuration, same references).
if RUN_NATURAL and PROFILE=='project':
    from sacrebleu.metrics import BLEU, CHRF
    refs={r['id']:r['target'] for r in workload}; quality=[]; translations=[]
    for k,grp in pred_df[(pred_df['mode']=='natural') & (pred_df.repeat==0)].groupby(keys):
        if len(grp)!=N_SENTENCES: continue
        texts=[tokenizer.decode(json.loads(s),skip_special_tokens=True,clean_up_tokenization_spaces=False).split('\n',1)[0].strip() for s in grp.token_ids]
        ref=[refs[i] for i in grp.id]; bleu=BLEU(); chrf=CHRF(word_order=0,beta=2,char_order=6)
        quality.append(dict(zip(keys,k))|dict(bleu=bleu.corpus_score(texts,[ref]).score,chrf=chrf.corpus_score(texts,[ref]).score,
                      bleu_signature=str(bleu.get_signature()),chrf_signature=str(chrf.get_signature())))
        translations.extend(dict(zip(keys,k))|dict(id=i,prediction=t,reference=r) for i,t,r in zip(grp.id,texts,ref))
    pd.DataFrame(quality).to_csv(OUT/'natural_quality.csv',index=False)
    pd.DataFrame(translations).to_csv(OUT/'natural_translations.csv',index=False)

## 7. Predictions and cache-memory validation
Use paired repeats for speedup and throughput ratios. The cache-memory prediction compares retained allocation minus returned sequence IDs against the theoretical cache size; the residual may include allocator rounding (512-byte blocks) and other returned objects. `requested_relative_error` repeats the comparison with requested (unrounded) bytes: when it is near zero but the allocated-memory error is not, the gap comes from the caching allocator reusing a larger block, not from the cache itself. Report both. The direct K/V tensor payload is asserted to equal the formula, which confirms the cache shape and dtype. Peak allocated memory is a different metric and should not be expected to equal cache bytes. Predictions are marked not evaluated if their required configurations did not complete.

In [ ]:
fixed=complete[complete['mode']=='fixed']
pair=repeat_df.merge(complete[keys],on=keys,how='inner')
on=pair[pair.use_cache]; off=pair[~pair.use_cache]
paired=on.merge(off,on=keys[:-1]+['repeat'],suffixes=('_on','_off'),validate='one_to_one')
paired['latency_speedup']=paired.batch_latency_ms_off/paired.batch_latency_ms_on
paired['throughput_ratio']=paired.tokens_per_s_on/paired.tokens_per_s_off
paired.to_csv(OUT/'paired_speedups.csv',index=False)
cache=raw[raw.use_cache].copy()
cache['payload_relative_error']=(cache.cache_bytes-cache.predicted_cache_bytes).abs()/cache.predicted_cache_bytes
cache['retained_minus_ids_bytes']=pd.to_numeric(cache.retained_minus_ids_bytes,errors='coerce')  # None on CPU
cache['allocator_residual_bytes']=cache.retained_minus_ids_bytes-cache.predicted_cache_bytes
cache['retained_relative_error']=cache.allocator_residual_bytes.abs()/cache.predicted_cache_bytes
# Diagnostic: if requested bytes match but allocated bytes do not, the gap is allocator block reuse, not the cache.
cache['requested_relative_error']=(pd.to_numeric(cache.requested_minus_ids_bytes,errors='coerce')-cache.predicted_cache_bytes).abs()/cache.predicted_cache_bytes
# Implementation check, not a prediction: K/V tensors must have exactly the formula's size.
assert (cache.payload_relative_error==0).all(), 'K/V tensor payload differs from 2*B*L*T*d*s; inspect cache layout.'
cache.to_csv(OUT/'cache_theory.csv',index=False)

def finite_or_none(x): return float(x) if pd.notna(x) and np.isfinite(x) else None
s=paired[(paired['mode']=='fixed')&(paired.max_new_tokens==64)&(paired.batch_size==1)].latency_speedup
b=on[(on['mode']=='fixed')&(on.max_new_tokens==64)]
x=b[b.batch_size==8].merge(b[b.batch_size==1],on='repeat',suffixes=('_8','_1'))
ratio=x.tokens_per_s_8/x.tokens_per_s_1
valid_cache=cache.merge(complete[keys],on=keys,how='inner')
err=finite_or_none(valid_cache.retained_relative_error.max()) if DEVICE.type=='cuda' else None
# Only project-profile comparisons can be evidence for the report hypotheses.
def outcome(value,threshold,op):
    if PROFILE!='project' or value is None: return 'not_evaluated'
    return 'supported' if op(value,threshold) else 'not_supported'
sv=finite_or_none(s.mean()) if len(s)==REPEATS else None
rv=finite_or_none(ratio.mean()) if len(ratio)==REPEATS else None
import operator
outcomes={'speedup_B1_G64':dict(value=sv,threshold=1.5,status=outcome(sv,1.5,operator.ge)),
          'cached_throughput_B8_over_B1_G64':dict(value=rv,threshold=2.,status=outcome(rv,2.,operator.ge)),
          'max_retained_memory_relative_error':dict(value=err,threshold=.10,status=outcome(err,.10,operator.le),
                                            complete_cache_configurations=int(complete.use_cache.sum()),
                                            intended_cache_configurations=sum(k[3] for k in configs))}
dump(OUT/'hypothesis_outcomes.json',outcomes); print(json.dumps(outcomes,indent=2))

## 8. Figures for the report
Error bars are ± one sample standard deviation across repeats, not confidence intervals. The main figures use fixed-length generation; the optional natural-decoding results are exported separately. Latency is per batch, while throughput is tokens/s over the same workload. For cache theory, overlapping points at the same shape are normal. Peak memory is incremental: it subtracts the memory allocated before the call (mostly the model weights). The figures use the report's style and size (one 0.24\textwidth subfigure each) and are saved as PDF for LaTeX and PNG.

In [ ]:
# Report-style figures: same look as the M0 figures in the report (STIX serif, 6.5/7 pt text,
# 1 pt open axes, faint grid, tab10 colours) at the size of one 0.24\textwidth subfigure.
# Each figure is saved as PDF (for LaTeX) and PNG. Error bars are ± 1 SD across repeats.
from matplotlib.ticker import FuncFormatter, LogLocator, NullLocator
plt.rcdefaults()
plt.rcParams.update({
    'font.family': 'STIXGeneral', 'mathtext.fontset': 'stix', 'pdf.fonttype': 42, 'ps.fonttype': 42,
    'font.size': 6.5, 'axes.labelsize': 7, 'xtick.labelsize': 6.5, 'ytick.labelsize': 6.5,
    'legend.fontsize': 6, 'legend.title_fontsize': 6, 'legend.frameon': False,
    'legend.handletextpad': .4, 'legend.borderaxespad': .2, 'legend.labelspacing': .3,
    'axes.linewidth': 1, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': .25, 'grid.linewidth': .8,
    'xtick.major.width': 1, 'ytick.major.width': 1, 'xtick.major.size': 3, 'ytick.major.size': 3,
    'lines.linewidth': 1, 'lines.markersize': 3, 'errorbar.capsize': 1.5,
    'figure.dpi': 200, 'savefig.dpi': 300,
})
PANEL = (1.62, 1.40)           # inches; one quarter-width subfigure in the report
REPORT_G = max(GEN_LENGTHS)    # the preregistered predictions are stated at 64 generated tokens
CACHE = {True: dict(color='tab:blue', marker='o', label='KV cache'),
         False: dict(color='tab:orange', marker='s', label='no cache')}
plain = FuncFormatter(lambda v, _: f'{v:g}')

def panel():
    fig, ax = plt.subplots(figsize=PANEL)
    for side in ('left', 'bottom'): ax.spines[side].set_position(('outward', 3))
    return fig, ax

def batch_axis(ax, log_y=True):
    ax.set_xscale('log', base=2); ax.set_xticks(BATCH_SIZES, labels=[str(b) for b in BATCH_SIZES])
    ax.xaxis.set_minor_locator(NullLocator()); ax.set_xlabel('Batch size')
    if log_y:
        ax.set_yscale('log'); lo, hi = ax.get_ylim()
        # Plain-number ticks at 1-2-5 steps on short ranges, 1-3 steps on long ones.
        ax.yaxis.set_major_locator(LogLocator(base=10, subs=(1, 2, 5) if hi / lo < 30 else (1, 3)))
        ax.yaxis.set_major_formatter(plain); ax.yaxis.set_minor_locator(NullLocator())

def finish(fig, name):
    fig.savefig(OUT / f'{name}.pdf', bbox_inches='tight', pad_inches=.02)
    fig.savefig(OUT / f'{name}.png', bbox_inches='tight', pad_inches=.02)
    plt.show(); plt.close(fig)

def by_cache(metric, ylabel, name, scale=1.0):
    d = fixed[fixed.max_new_tokens == REPORT_G]
    if d.empty or d[f'{metric}_mean'].isna().all(): return
    fig, ax = panel()
    for c in (True, False):
        s = d[d.use_cache == c].sort_values('batch_size')
        if s.empty: continue
        y, e = s[f'{metric}_mean'] * scale, s[f'{metric}_std'].fillna(0) * scale
        ax.errorbar(s.batch_size, y, yerr=e, fmt='none', ecolor=CACHE[c]['color'], elinewidth=.8)
        ax.plot(s.batch_size, y, **CACHE[c])  # separate so legend entries show no error-bar caps
    batch_axis(ax); ax.set_ylabel(ylabel); ax.legend()
    finish(fig, name)

# (1) Throughput and (2) latency against batch size, with and without the cache.
by_cache('tokens_per_s', 'Generated tokens/s', 'q3b_throughput')
by_cache('batch_latency_ms', 'Latency per batch (s)', 'q3b_latency', scale=1e-3)
# (3) Incremental peak memory (excludes the model weights already on the GPU).
if DEVICE.type == 'cuda':
    by_cache('peak_increment_mib', 'Peak extra memory (MiB)', 'q3b_peak_memory')  # above the weights

# (4) Speedup from the cache at each generation length (no colour: colour means cache on/off above).
if not paired.empty:
    fig, ax = panel()
    lengths = sorted(paired[paired['mode'] == 'fixed'].max_new_tokens.unique())
    for g in lengths:
        z = (paired[(paired['mode'] == 'fixed') & (paired.max_new_tokens == g)]
             .groupby('batch_size').latency_speedup.agg(['mean', 'std']).reset_index())
        top = g == REPORT_G; col = '0.15' if top else '0.55'
        ax.errorbar(z.batch_size, z['mean'], yerr=z['std'].fillna(0), fmt='none', ecolor=col, elinewidth=.8)
        ax.plot(z.batch_size, z['mean'], color=col, marker='o', mfc=col if top else 'white',
                ls='-' if top else '--', label=str(g))
    ax.axhline(1, color='0.3', lw=.8, ls=':')
    batch_axis(ax); ax.set_ylabel('Speedup from cache (×)')
    ax.legend(title='Generated tokens')
    finish(fig, 'q3b_speedup')

# (5) Cache memory against theory, laid out like the M0 memory figure: every cached call.
if DEVICE.type == 'cuda' and not valid_cache.empty:
    d = valid_cache[valid_cache['mode'] == 'fixed']
    if not d.empty:
        fig, ax = panel()
        x = d.predicted_cache_bytes / MIB
        ax.scatter(x, d.peak_increment_bytes / MIB, s=6, color='tab:blue', marker='o', alpha=.45, lw=0,
                   label='peak during call')
        ax.scatter(x, d.retained_minus_ids_bytes / MIB, s=7, color='tab:orange', marker='s', alpha=.7, lw=0,
                   label='held after call')
        top = x.max() * 1.03
        ax.plot([0, top], [0, top], ls='--', color='0.3', lw=1.2, label='prediction')
        ax.set_xlim(left=0); ax.set_ylim(bottom=0)
        ax.set(xlabel='Predicted KV cache (MiB)', ylabel='Measured memory (MiB)')
        ax.legend(fontsize=5.5, loc='upper left', markerscale=1.6)
        finish(fig, 'q3b_memory')
print('Saved report figures (PDF + PNG) to', OUT)

## 9. Export and reporting notes
For Q3(b), cite `summary_complete.csv`, `paired_speedups.csv`, `cache_theory.csv`, `hypothesis_outcomes.json`, and the figures. Report the exact GPU, precision, attention backend, checkpoint, prompt width, workload size, and fixed generation length. If some configurations fail, report those OOMs and do not extrapolate. If token-ID agreement is imperfect, describe it as observed numerical/decoding variation; do not claim caching changes the mathematical model. Fixed-length outputs should not be scored as translations. If using the optional natural sweep, report useful tokens and decode slots separately and note early stopping. A single GPU and a single prompt cohort limit external validity; repeated timings measure run variability, not a population of GPUs.

In [ ]:
archive=OUT.parent/f'{OUT.name}_report.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob('*'):
        if p.is_file(): z.write(p,p.relative_to(OUT))
print('Report archive:',archive.resolve())
print('Configuration and predictions:',OUT/'preregistration.json')
print('Primary table:',OUT/'summary_complete.csv')
# Optional in Colab: from google.colab import files; files.download(str(archive))